# Building Golden Datasets from Production Traces with Falcon AI

[Read the cookbook](https://docs.futureagi.com/docs/cookbook/falcon-ai/eval-datasets-from-traces). Trace six synthetic support emails, label the five reviewed cases and upload a snapshot with trace IDs. Use a fresh Python 3.12 kernel and working directory. You need Future AGI and OpenAI credentials. Hosted calls can consume credits.

## Install tracing

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "fi-instrumentation-otel", "traceai-openai", "openai", "virtualenv"], check=True)

In [ ]:
from getpass import getpass
import os

for name in ('FI_API_KEY', 'FI_SECRET_KEY', 'OPENAI_API_KEY'):
    if not os.environ.get(name):
        os.environ[name] = getpass(f"{name}: ")
    if not os.environ[name].strip():
        raise ValueError(f"{name} is required")

# For self-hosting, set this to your collector before running the cell.
os.environ.setdefault("FI_TRACES_URL", "https://api.futureagi.com")

## Add tracing to your agent

In [ ]:
from fi_instrumentation import register, FITracer
from fi_instrumentation.fi_types import ProjectType
from traceai_openai import OpenAIInstrumentor

trace_provider = register(
    verbose=False,
    project_type=ProjectType.OBSERVE,
    project_name="ridgeline-email-triage-demo",
)
OpenAIInstrumentor().instrument(tracer_provider=trace_provider)
tracer = FITracer(trace_provider.get_tracer("ridgeline-email-triage-demo"))

In [ ]:
import csv
from uuid import UUID
from openai import OpenAI
from opentelemetry.trace import get_current_span

client = OpenAI()

@tracer.agent(name="triage_email")
def triage_email(email_text):
    reply = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Route support email to urgent, billing, technical, general, or spam. Reply with just the label."},
            {"role": "user", "content": email_text},
        ],
    ).choices[0].message.content
    if not reply:
        raise ValueError("Empty classification")
    return str(UUID(int=get_current_span().get_span_context().trace_id)), reply.strip()

# Reviewer's routing labels. Only email_text goes into the classifier.
cases = [
    ("The checkout system is down for every store.", "urgent"),
    ("I see two charges for my sleeping bag. Please refund the duplicate.", "billing"),
    ("The app crashes when I open my saved hiking routes.", "technical"),
    ("What time does your shop open on Sunday?", "general"),
    ("Buy bulk advertising lists now! Limited offer!", "spam"),
    ("My receipt is wrong and my login no longer works.", "NEEDS_REVIEW"),
]
with open("triage-traces.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["email_text", "predicted_category", "expected_category", "trace_id"])
    for email, expected in cases:
        trace_id, label = triage_email(email)
        writer.writerow([email, label, expected, trace_id])
        print(trace_id, label)
trace_provider.force_flush()

## Explore failures with Falcon AI

Open the tracing project in the dashboard. Ask Falcon AI which categories the classifier assigned. Match the six trace IDs to `triage-traces.csv`.

## Build the dataset and add ground truth

In Falcon AI, use `/build-dataset` with those six IDs. Ask for `email_text`, `predicted_category`, and `trace_id`. Review the labels yourself; keep the multi-issue email as `NEEDS_REVIEW` rather than treating its prediction as the answer.

## Lock in a baseline eval

In the dashboard, use `/run-evaluations` with `equals` on the two category columns for the five reviewed emails. Or use the scripted snapshot below; it does not require completing the dashboard steps.

## Save the reviewed example from Python

Agent Learning Kit and the platform SDK both install `fi` modules. Install them into separate environments. The tracing kernel stays separate from both.

In [ ]:
for folder, package in [(".judges", "agent-learning-kit==0.3.7"), (".datasets", "futureagi==0.7.0")]:
    subprocess.run([sys.executable, "-m", "virtualenv", "--quiet", folder], check=True)
    subprocess.run([f"{folder}/bin/python", "-m", "pip", "install", "--quiet", package], check=True)

In [ ]:
%%writefile score_triage.py
import csv
from fi.evals import Evaluator

evaluator = Evaluator()
with open("triage-traces.csv", newline="") as f:
    rows = list(csv.DictReader(f))
for row in rows:
    row.update(verdict="Pending review", eval_id="")
    if row["expected_category"] == "NEEDS_REVIEW":
        continue
    result = evaluator.evaluate(eval_templates="equals", inputs={
        "text": row["predicted_category"], "expected_text": row["expected_category"],
    }).eval_results[0]
    if result.output not in ("Passed", "Failed"):
        raise RuntimeError(result.reason or "Missing exact-match verdict")
    row.update(verdict=result.output, eval_id=result.eval_id)
scored = [row for row in rows if row["eval_id"]]
print(f"{sum(row['verdict'] == 'Passed' for row in scored)} of {len(scored)} reviewed emails matched")
with open("reviewed-triage.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)

In [ ]:
print(subprocess.run([".judges/bin/python", "score_triage.py"], check=True, stdout=subprocess.PIPE, text=True).stdout.strip())

In [ ]:
%%writefile upload_triage.py
from uuid import uuid4
from fi.datasets import Dataset, DatasetConfig
from fi.utils.types import ModelTypes

name = f"reviewed-email-triage-{uuid4().hex[:8]}"
dataset = Dataset(dataset_config=DatasetConfig(name=name, model_type=ModelTypes.GENERATIVE_LLM))
dataset.create(source="reviewed-triage.csv")
print(dataset.dataset_config.id)

In [ ]:
print(subprocess.run([".datasets/bin/python", "upload_triage.py"], check=True, stdout=subprocess.PIPE, text=True).stdout.strip())

Open the newly created dataset in your workspace. Check its six trace IDs against the CSV; only the five reviewed cases should have an eval ID.